# Library 05 – Network measures

The same measures, with the same settings, are used for Test, ODI and T20I. Full reasons for each choice are in
`docs/methods_decisions.md`.

**One rule about weights.** The edge weight is a *strength*: more runs = a stronger tie. Shortest-path
algorithms read edge values as *distances*: a bigger value = further away. So we use weights only where a
strength makes sense:

| Uses weights (runs) | Ignores weights (every tie = 1 step) |
|---|---|
| strength, eigenvector centrality, Louvain communities | betweenness, closeness, path length, diameter, clustering, k-core, redundancy, constraint |

Closeness, eigenvector centrality and path statistics are computed on the largest connected component. In our
data each network has only one component, so that is the whole network.

Needs: `lib_00_settings`.

In [ ]:
def largest_component(partnership_graph: nx.Graph) -> nx.Graph:
    """Sub-graph made of the largest connected component."""
    biggest_node_set = max(nx.connected_components(partnership_graph), key=len)
    return partnership_graph.subgraph(biggest_node_set).copy()


def louvain_communities(partnership_graph: nx.Graph, seed: int = RANDOM_SEED) -> list[set[str]]:
    """Weighted Louvain communities, sorted by size (largest first) so numbering is stable."""
    communities = nx.community.louvain_communities(partnership_graph, weight="weight",
                                                   resolution=LOUVAIN_RESOLUTION, seed=seed)
    return sorted(communities, key=lambda community: (-len(community), min(community)))


def degree_centralization(partnership_graph: nx.Graph) -> float:
    """Freeman degree centralisation: 0 when all degrees are equal, 1 for a star."""
    node_count = partnership_graph.number_of_nodes()
    if node_count < 3:
        return float("nan")
    degrees = [degree for _, degree in partnership_graph.degree()]
    return sum(max(degrees) - degree for degree in degrees) / ((node_count - 1) * (node_count - 2))

## Small-world comparison

We compare our network with random graphs that have the same number of nodes and edges (G(n, m) graphs).
`sigma = (C / C_random) / (L / L_random)`, where C is the average clustering and L the average shortest-path
length. A value above 1 means "more clustered than random, with similar path lengths".

In [ ]:
def small_world_comparison(partnership_graph: nx.Graph, number_of_graphs: int = NUMBER_OF_RANDOM_GRAPHS,
                           seed: int = RANDOM_SEED) -> dict[str, float]:
    """Clustering and path length of the network against seeded random G(n, m) graphs."""
    main_component = largest_component(partnership_graph)
    node_count, edge_count = main_component.number_of_nodes(), main_component.number_of_edges()
    observed_clustering = nx.average_clustering(main_component)
    observed_path_length = nx.average_shortest_path_length(main_component)
    random_generator = np.random.default_rng(seed)
    random_clusterings, random_path_lengths = [], []
    for _ in range(number_of_graphs):
        random_graph = nx.gnm_random_graph(node_count, edge_count, seed=int(random_generator.integers(0, 2**31 - 1)))
        random_graph = largest_component(random_graph)
        random_clusterings.append(nx.average_clustering(random_graph))
        random_path_lengths.append(nx.average_shortest_path_length(random_graph))
    mean_random_clustering = float(np.mean(random_clusterings))
    mean_random_path_length = float(np.mean(random_path_lengths))
    return {
        "C_random": mean_random_clustering,
        "L_random": mean_random_path_length,
        "C_ratio": observed_clustering / mean_random_clustering,
        "L_ratio": observed_path_length / mean_random_path_length,
        "small_world_sigma": (observed_clustering / mean_random_clustering) / (observed_path_length / mean_random_path_length),
    }


def louvain_stability(partnership_graph: nx.Graph, number_of_runs: int = LOUVAIN_STABILITY_RUNS) -> dict[str, float]:
    """How much the Louvain result changes over different seeds (0 .. number_of_runs - 1)."""
    modularity_values, community_counts = [], []
    for seed in range(number_of_runs):
        communities = louvain_communities(partnership_graph, seed=seed)
        modularity_values.append(nx.community.modularity(partnership_graph, communities, weight="weight"))
        community_counts.append(len(communities))
    return {"louvain_Q_min": min(modularity_values), "louvain_Q_max": max(modularity_values),
            "louvain_k_min": min(community_counts), "louvain_k_max": max(community_counts)}

## Node-level measures

In [ ]:
def node_measures(partnership_graph: nx.Graph) -> pd.DataFrame:
    """One row per batter with every node-level measure, sorted by strength."""
    main_component = largest_component(partnership_graph)
    node_count = partnership_graph.number_of_nodes()

    degree_of = dict(partnership_graph.degree())                                # distinct partners
    strength_of = dict(partnership_graph.degree(weight="weight"))               # total partnership runs
    betweenness_of = nx.betweenness_centrality(partnership_graph, weight=None, normalized=True)
    closeness_of = nx.closeness_centrality(main_component)                      # unweighted, main component
    eigenvector_of = nx.eigenvector_centrality(main_component, weight="weight", max_iter=5000, tol=1e-10)
    clustering_of = nx.clustering(partnership_graph)                            # unweighted local clustering
    core_number_of = nx.core_number(partnership_graph)                          # k-core shell of each node
    effective_size_of = nx.effective_size(partnership_graph)                    # Burt, binary ties
    constraint_of = nx.constraint(partnership_graph)                            # Burt, binary ties
    community_of = {player: number for number, community in enumerate(louvain_communities(partnership_graph), start=1)
                    for player in community}

    measure_rows = []
    for player_id, node_attributes in partnership_graph.nodes(data=True):
        degree = degree_of[player_id]
        measure_rows.append({
            "format": node_attributes["format"],
            "player_id": player_id,
            "player_name": node_attributes["player_name"],
            "matches_batted": node_attributes["matches_batted"],
            "innings_batted": node_attributes["innings_batted"],
            "partnership_observations": node_attributes["partnership_observations"],
            "first_date": node_attributes.get("first_date", ""),
            "last_date": node_attributes.get("last_date", ""),
            "median_entry_partnership": node_attributes.get("median_entry_partnership", float("nan")),
            "degree": degree,
            "degree_centrality": degree / (node_count - 1),
            "strength": int(strength_of[player_id]),
            "strength_per_innings": strength_of[player_id] / node_attributes["innings_batted"],
            "betweenness": betweenness_of[player_id],
            "in_lcc": player_id in main_component,
            "closeness_lcc": closeness_of.get(player_id, float("nan")),
            "eigenvector_lcc": eigenvector_of.get(player_id, float("nan")),
            "clustering": clustering_of[player_id],
            "core_number": core_number_of[player_id],
            "community": community_of[player_id],
            "effective_size": effective_size_of[player_id],
            "redundancy": degree - effective_size_of[player_id],
            "efficiency": effective_size_of[player_id] / degree if degree else float("nan"),
            "constraint": constraint_of[player_id],
        })
    return (pd.DataFrame(measure_rows).sort_values(["strength", "player_id"], ascending=[False, True])
            .reset_index(drop=True))

## Network-level measures

In [ ]:
def network_summary(partnership_graph: nx.Graph, node_measure_table: pd.DataFrame) -> dict[str, float]:
    """Whole-network description for one format."""
    node_count, edge_count = partnership_graph.number_of_nodes(), partnership_graph.number_of_edges()
    component_sizes = sorted((len(component) for component in nx.connected_components(partnership_graph)), reverse=True)
    main_component = largest_component(partnership_graph)
    degrees = [degree for _, degree in partnership_graph.degree()]
    strengths = [strength for _, strength in partnership_graph.degree(weight="weight")]
    edge_weights = [attributes["weight"] for _, _, attributes in partnership_graph.edges(data=True)]
    partnerships_per_edge = [attributes["partnership_count"] for _, _, attributes in partnership_graph.edges(data=True)]
    communities = louvain_communities(partnership_graph)
    core_number_of = nx.core_number(partnership_graph)
    highest_core = max(core_number_of.values())
    top_ten_percent_edges = max(1, round(0.1 * edge_count))

    summary = {
        "nodes": node_count, "edges": edge_count,
        "density": nx.density(partnership_graph),              # 2m / (n (n - 1)), no self-loops
        "components": len(component_sizes), "lcc_nodes": component_sizes[0],
        "lcc_share": component_sizes[0] / node_count, "connected": int(len(component_sizes) == 1),
        "degree_mean": float(np.mean(degrees)), "degree_median": float(np.median(degrees)),
        "degree_min": min(degrees), "degree_max": max(degrees), "degree_sd": float(np.std(degrees, ddof=1)),
        "degree_centralization": degree_centralization(partnership_graph),
        "strength_mean": float(np.mean(strengths)), "strength_median": float(np.median(strengths)),
        "strength_max": max(strengths),
        "edge_weight_total": int(sum(edge_weights)), "edge_weight_mean": float(np.mean(edge_weights)),
        "edge_weight_median": float(np.median(edge_weights)), "edge_weight_max": max(edge_weights),
        "edges_single_observation_share": sum(1 for count in partnerships_per_edge if count == 1) / edge_count,
        "top10pct_edges_weight_share": float(np.sort(edge_weights)[::-1][:top_ten_percent_edges].sum() / sum(edge_weights)),
        "avg_clustering": nx.average_clustering(partnership_graph),
        "transitivity": nx.transitivity(partnership_graph),
        "avg_path_length_lcc": nx.average_shortest_path_length(main_component),
        "diameter_lcc": nx.diameter(main_component),
        "degree_assortativity": nx.degree_assortativity_coefficient(partnership_graph),
        "max_core_k": highest_core,
        "max_core_nodes": sum(1 for core in core_number_of.values() if core == highest_core),
        "louvain_communities": len(communities),
        "louvain_modularity": nx.community.modularity(partnership_graph, communities, weight="weight"),
        "louvain_sizes": "|".join(str(len(community)) for community in communities),
        "mean_effective_size": float(node_measure_table["effective_size"].mean()),
        "mean_efficiency": float(node_measure_table["efficiency"].mean()),
        "mean_constraint": float(node_measure_table["constraint"].mean()),
        "median_innings_batted": statistics.median(node_measure_table["innings_batted"]),
    }
    summary.update(small_world_comparison(partnership_graph))
    summary.update(louvain_stability(partnership_graph))
    return summary